In [ ]:
%matplotlib widget

from blackjack_py import ProbabilisticRankShoe


from blackjack.blackjack_round import BJRound, BJStage, BJRules
from blackjack.actions import PlayerAction, DealerAction
from blackjack.cards import Card, Rank
import numpy as np
import time
from datetime import datetime
import os
import tqdm
from collections import deque
# from blackjack.shoe import ProbabilisticRankShoe
import time
from blackjack.floor_ceil_node import FloorCeilNode, SplitNode, DecisionNode, DealerCheckBJNode
from blackjack.dealer_sim import run_dealer_cards_simulation
import datetime
import matplotlib.pyplot as plt
from blackjack.tree_utils import iterate_nodes_by_levels
import os
from blackjack import abstract_node


In [ ]:
prsh = ProbabilisticRankShoe.seeded(1,0)

In [ ]:
rules = BJRules(
    dealer_checks_blackjack=True,
    dealer_hits_soft_17=False,
    allow_late_surrender=False,
    allow_early_surrender_on_ten=False,
    allow_early_surrender_on_ace=False,
    allow_early_surrender_on_all=False,
    dealer_shows_card_on_surrender=False,
    allow_insurance_vs_ace=True,
    natural_blackjack_payout=3/2,
    surrender_payout=1/2,
    insurance_payout=2/1,
    max_splits_allowed=1,
    allow_action_on_split_aces=True,
    allow_double_after_split=True,
    allow_double_on_soft=True,
    allow_split_different_tens=True
)

In [ ]:
from collections import defaultdict
from collections import Counter
from blackjack.abstract_node import FloorCeilValueNode, ValueNode
from blackjack.tree_utils import get_nodes_by_levels


def log_tree_structure(root_node):
    nodes_dict = get_nodes_by_levels(root_node)
    print(f"Total: {sum(len(nodes) for nodes in nodes_dict.values())} nodes")
    for lvl, nodes in nodes_dict.items():
        stage_count = count_stages(nodes)
        print(f"Level {lvl}: {len(nodes)} nodes", stage_count)


def count_stages(node_list):
    stage_counter = Counter()
    for node in node_list:
        stage_name = node.__class__.__name__

        if hasattr(node, "bj_round") and node.bj_round is not None:
            stage = node.bj_round.get_stage()
            child_placeholders = [ch for ch in node.children if isinstance(ch, FloorCeilValueNode)]
            if stage == BJStage.PLAYER_CARD:
                if len(child_placeholders) > 0:
                    stage_name += "_PLAYER_CARD_PARTIAL"
                else:
                    stage_name += "_PLAYER_CARD_FULL"
            elif isinstance(node, DecisionNode):
                stage_name += "_PENDING" if node.decision_choice is None else "_DECIDED"
            else:
                stage_name += "_" + stage.name
        
        stage_counter[stage_name] += 1
    return stage_counter

In [ ]:
main_actions_list = [
    PlayerAction.HIT,
    PlayerAction.STAND,
    PlayerAction.DOUBLE,
    PlayerAction.SPLIT
]


def best_action_hard_vs_dealer(hard_value, dealer_card):
    if hard_value < 5 or hard_value > 19:
        raise ValueError("Invalid hard value")

    if hard_value >= 12:
        player_card_0 = 10
    else:
        player_card_0 = 2
    
    player_card_1 = hard_value - player_card_0 

    cards = [
        Card(Rank.from_value(player_card_0)),
        Card(Rank.from_value(player_card_1)),
        Card(Rank.from_value(dealer_card))  
    ]
    
    bj_round = BJRound(rules)
    shoe = ProbabilisticRankShoe(8)
    bj_round.start_round(10)

    bj_round.take_card(cards[0])
    bj_round.take_card(cards[1])
    bj_round.take_card(cards[2])

    shoe.burn_rank_value(cards[0].rank_value())
    shoe.burn_rank_value(cards[1].rank_value())
    shoe.burn_rank_value(cards[2].rank_value())

    root_node = FloorCeilNode(bj_round, shoe, monte_carlo_depth=6)
    root_node.build_tree()
    
    expected_value_0 = root_node.get_value()
    expected_value_1 = None

    main_action = None
    insurance_action = None
    action_node = root_node
    while main_action is None:
        stage = action_node.bj_round.get_stage()
        if stage == BJStage.DEALER_CHECK_BJ:
            # find branch where dealer doesn't have bj
            for ch in action_node.children:
                if ch.last_action == DealerAction.CONFIRM_NO_BLACKJACK:
                    action_node = ch
                    break
        elif stage == BJStage.PLAYER_OFFERED_INSURANCE:
            child_idx = np.argmax(action_node.children_prob)
            action_node = action_node.children[child_idx]
            insurance_action = action_node.last_action
        elif stage == BJStage.PLAYER_ACTION:
            action_child_idx = np.argmax(action_node.children_prob)
            after_action_node = action_node.children[action_child_idx]
            main_action = after_action_node.bj_round.last_action
            expected_value_1 = after_action_node.get_value()
        else:
            raise RuntimeError(f"Unexpected stage {stage}")
    
    return main_action, insurance_action, expected_value_0, expected_value_1

In [ ]:
def inspect(root_node):
    for target_lvl in range(5):
        print(f"Lvl {target_lvl}")
        for lvl, ch in iterate_nodes_by_levels(root_node):
            if lvl < target_lvl:
                continue
            if lvl > target_lvl:
                break
            
            # Special handling for level 4
            if target_lvl == 4 and not isinstance(ch.parent, SplitNode):
                continue
            
            # Calculate values
            if target_lvl == 4 and isinstance(ch.parent, SplitNode):
                floor_val = 2 * ch.get_floor_value()
                val = 2 * ch.get_value()
                ceil_val = 2 * ch.get_ceil_value()
            else:
                floor_val = ch.get_floor_value()
                try:
                    val = ch.get_value()
                except:
                    val = np.nan
                ceil_val = ch.get_ceil_value()
            
            # Determine action label
            if isinstance(ch, ValueNode):
                action = "VN"
            elif isinstance(ch, FloorCeilValueNode):
                action = "FC_VN"
            elif isinstance(ch, DealerCheckBJNode):
                action = "ACCEPT_INSURANCE" if ch.took_insurance else "DECLINE_INSURANCE"
            elif isinstance(ch, SplitNode):
                action = "SPLIT"
            elif target_lvl == 4 and isinstance(ch.parent, SplitNode):
                action = ch.bj_round.player_hands[0][1]
            else:
                if ch.bj_round is None:
                    print(type(ch))
                event = ch.bj_round.last_action
                if event is not None:
                    action = event.value
                else:
                    event = ch.bj_round.last_card
                    action = str(event)
                
                # Add decision choice for level 2
                if target_lvl == 2 and hasattr(ch, 'decision_choice') and ch.decision_choice is not None:
                    action += f"({ch.decision_choice.value})"
            
            print(f"{action} value gap: [ {floor_val:.02f} {val:.02f} {ceil_val:.02f} ]")

    print()


In [ ]:
bj_round = BJRound(rules)
shoe = ProbabilisticRankShoe(8)
bj_round.start_round(10)

cards = [
    Card(Rank.ACE),
    Card(Rank.ACE),
    Card(Rank.SIX) 
]

# cards = [
#     Card(Rank.TEN),
#     Card(Rank.SIX),
#     Card(Rank.ACE)  
# ]

cards = [c.rank_value() for c in cards]

bj_round.take_card(cards[0])
bj_round.take_card(cards[1])
bj_round.take_card(cards[2])

shoe.burn_rank_value(cards[0])
shoe.burn_rank_value(cards[1])
shoe.burn_rank_value(cards[2])

root_node_fe1 = DecisionNode(
    bj_round,
    shoe,
    max_hand_size_full_enum=1,
    n_dealer_sim_runs=1
)


root_node_fe3 = DecisionNode(
    bj_round,
    shoe,
    max_hand_size_full_enum=3,
    n_dealer_sim_runs=1
)

print(str(bj_round))

In [ ]:
def build_tree(root_node):
    i = 0
    while True:
        t0 = time.time()
        root_node.build_tree_layer(depth=i)
        t1 = time.time()
        seconds = np.round(t1 - t0)
        dt_whole = datetime.timedelta(seconds=seconds)
        print(f"Depth {i} built in {dt_whole} ({seconds} s)")
        if root_node.tree_completed():
            print("Tree completed")
            break
        i += 1

In [ ]:
abstract_node.total_sim_time = 0
t0 = time.time()
build_tree(root_node_fe1)
t1 = time.time()
print(f"Simulation time {abstract_node.total_sim_time} seconds ")
print(f"Time taken to build tree for root_node_fe3: {t1 - t0} seconds")
print(f"{abstract_node.total_sim_time / (t1 - t0) * 100:.02f}%")

In [ ]:
for i in range(100):
    abstract_node.total_sim_time = 0
    t0 = time.time()
    root_node_fe1.convert_to_full_up_to_depth(depth=i)
    t1 = time.time()
    print(f"Depth {i} Simulation time {abstract_node.total_sim_time} seconds ")
    print(f"Gap {root_node_fe1.get_ceil_value()}, {root_node_fe1.get_floor_value()}")
    print(f"Time taken to build tree for root_node_fe3: {t1 - t0} seconds")
    print(f"{abstract_node.total_sim_time / (t1 - t0) * 100:.02f}%")

    if root_node_fe1.get_ceil_value() - root_node_fe1.get_floor_value() < 0.01:
        break

In [ ]:
abstract_node.total_sim_time = 0
t0 = time.time()
build_tree(root_node_fe3)
t1 = time.time()
print(f"Simulation time {abstract_node.total_sim_time} seconds ")
print(f"Time taken to build tree for root_node_fe3: {t1 - t0} seconds")
print(f"{abstract_node.total_sim_time / (t1 - t0) * 100:.02f}%")

In [ ]:
for i in range(100):
    abstract_node.total_sim_time = 0
    t0 = time.time()
    root_node_fe3.convert_to_full_up_to_depth(depth=i)
    t1 = time.time()
    print(f"Depth {i} Simulation time {abstract_node.total_sim_time} seconds ")
    print(f"Gap {root_node_fe3.get_ceil_value()}, {root_node_fe3.get_floor_value()}")
    print(f"Time taken to build tree for root_node_fe3: {t1 - t0} seconds")
    print(f"{abstract_node.total_sim_time / (t1 - t0) * 100:.02f}%")

    if root_node_fe3.get_ceil_value() - root_node_fe3.get_floor_value() < 0.01:
        break

In [ ]:
print(root_node_fe1.get_floor_value(), root_node_fe1.get_value(), root_node_fe1.get_ceil_value())
print(root_node_fe3.get_floor_value(), root_node_fe3.get_value(), root_node_fe3.get_ceil_value())


In [ ]:
# 7.19-7.20 - depth 2
# 9.136-9.137
# 9.163-9.165

In [ ]:
abstract_node.total_sim_time

In [ ]:
log_tree_structure(root_node_fe1)
inspect(root_node_fe1)

In [ ]:
log_tree_structure(root_node_fe3)

inspect(root_node_fe3)

In [ ]:
print(root_node_fe1.get_floor_value(), root_node_fe1.get_value(), root_node_fe1.get_ceil_value())

In [ ]:
inspect(root_node_fe1)

In [ ]:
inspect(root_node_fe3)

In [ ]:
# lets compare split->card 8 node

In [ ]:
split_8_node_fe1 = root_node_fe1.children[1].children[1].children[3].children[6]
split_8_node_fe3 = root_node_fe3.children[1].children[1].children[3].children[6]

In [ ]:
log_tree_structure(split_8_node_fe1)
log_tree_structure(split_8_node_fe3)

In [ ]:
inspect(split_8_node_fe1)

In [ ]:
inspect(split_8_node_fe3)

In [ ]:
round = split_8_node_fe3.bj_round.copy()
round.take_action(PlayerAction.STAND)

results_100 = []
results_1000 = []
results_100_2 = []
results_100_3 = []
results_10_3 = []


n_runs = 3600

t0 = time.time()
for i in range(n_runs // 2):
    result_10_3 = run_dealer_cards_simulation_alt(
        round, split_8_node_fe3.shoe, n_dealer_sim_runs=10, n_full_sample=3
    )
    results_10_3.append(result_10_3)

t1 = time.time()
for i in range(n_runs // 2):
    result_100_3 = run_dealer_cards_simulation_alt(
        round, split_8_node_fe3.shoe, n_dealer_sim_runs=10, n_full_sample=3
    )
    results_100_3.append(result_100_3)

t2 = time.time()
for i in range(n_runs // 6):
    result_100_2 = run_dealer_cards_simulation_alt(
        round, split_8_node_fe3.shoe, n_dealer_sim_runs=100, n_full_sample=2
    )
    results_100_2.append(result_100_2)

t3 = time.time()
for i in range(n_runs // 12):
    result_1000 = run_dealer_cards_simulation_alt(
        round, split_8_node_fe3.shoe, n_dealer_sim_runs=1000, n_full_sample=1
    )
    results_1000.append(result_1000)

t4 = time.time()
for i in range(n_runs):
    result_100 = run_dealer_cards_simulation_alt(
        round, split_8_node_fe3.shoe, n_dealer_sim_runs=100
    )
    results_100.append(result_100)
t5 = time.time()

print(f"10 runs 3 levels time: {t1 - t0} s")
print(f"100 runs 3 levels time: {t2 - t1} s")
print(f"100 runs 2 levels time: {t3 - t2} s")
print(f"1000 runs time: {t4 - t3} s")
print(f"100 runs time: {t5 - t4} s")


In [ ]:
f, ax = plt.subplots(1,1)
# ax.hist(results_100, label="sim 100", alpha=0.5, density=True)
# ax.hist(results_1000, label="sim 1000",  alpha=0.5, density=True)
ax.hist(results_100_2, label="sim 100 2 lvl",  alpha=0.5, density=True)
ax.hist(results_100_3, label="sim 100 3 lvl",  alpha=0.5, density=True)
ax.hist(results_10_3, label="sim 10 3 lvl",  alpha=0.5, density=True)
ax.legend()

In [ ]:
[ch.get_value() for ch in root_node.children]

In [ ]:
root_node.children_events

In [ ]:
log_tree_structure(root_node)

In [ ]:
from blackjack.floor_ceil_node import DealerCheckBJNode

ceil = []
ev = []
floor = []
runtime = []

t_start = time.time()

root_node = root_node_fe1

for i in range(100):
    t0 = time.time()
    children_changed = root_node.convert_to_full_up_to_depth(depth=i)
    t1 = time.time()
    seconds = np.round(t1 - t0)

    floor_val = root_node.get_floor_value()
    val = root_node.get_value()
    ceil_val = root_node.get_ceil_value()

    floor.append(floor_val)
    ev.append(val)
    ceil.append(ceil_val)
    runtime.append(t1 - t_start)

    dt_whole = datetime.timedelta(seconds=seconds)
    print(f"Depth {i} converted to full enumeration in {dt_whole} ({seconds} s)")
    print(f"Value gap: [{floor_val:.02f} {val:.02f} {ceil_val:.02f}]")

    if (ceil_val - floor_val) < 0.001:
        break
    # log_tree_structure(root_node)

In [ ]:
f, ax = plt.subplots(figsize=(10, 6))
ax.plot(runtime, floor, label="Floor Value", linestyle='--', color="grey" )
ax.plot(runtime, ev, label="Expected Value", color="red")
ax.plot(runtime, ceil, label="Ceil Value", linestyle='--', color='grey')

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.legend()

In [ ]:
# starting from initial limit of 1 to depth 5 is a reasonable approach

In [ ]:
refuse_insurance_node = root_node.children[1]
print(refuse_insurance_node.children_prob)

In [ ]:
print([ch.get_floor_value() for ch in refuse_insurance_node.children])
print([ch.get_ceil_value() for ch in refuse_insurance_node.children])

In [ ]:
refuse_insurance_node.floor_value, refuse_insurance_node.ceil_value

In [ ]:
refuse_insurance_node.recompute_tree_value()

In [ ]:
refuse_insurance_node.floor_value, refuse_insurance_node.ceil_value

In [ ]:
lvl25 = None
for lvl, node in iterate_nodes_by_levels(root_node):
    if lvl == 24 and isinstance(node, DecisionNode):
        lvl25 = node
        break

In [ ]:
print(str(lvl25.bj_round))

In [ ]:
# with open("logs/game_tree.json", "w") as f:
#     game_tree_to_json(f, root_node)

In [ ]:
print(root_node.get_value())
print(root_node.children_prob)
print(root_node.children_events)
print([f"{ch.get_value():.2f}" for ch in root_node.children])

In [ ]:
node = root_node.children[-1].children[7]
print("decision", node.decision_choice)
print(str(node.bj_round))
print("value = ", node.get_value())
print(node.children_prob)
print(node.children_events)

node.decision_action = None
for ch in node.children:
    ch.decision_action = None
    
print([f"{ch.get_value():.2f}" for ch in node.children])
print([f"{ch.get_ceil_value():.2f}" for ch in node.children])
print([f"{ch.get_floor_value():.2f}" for ch in node.children])

In [ ]:
node.update_decision()

In [ ]:
node.children[1].get_value()

In [ ]:
print(node.children[1].cards_sampled) 
print(node.children[1].cards_not_sampled)
print(node.children[1].cards_21)
print(node.children[1].cards_bust)

In [ ]:
print(set(node.children[1].children_prob))

In [ ]:
node_child = node.children[1]

print(node_child.children_prob)
print(node_child.children_events)
print([f"{ch.get_value():.2f}" for ch in node_child.children])


In [ ]:
np.array(node_child.children_prob).dot(np.array([ch.get_value() for ch in node_child.children]))

In [ ]:
for ch, ch_event in zip(node.children, node.children_events):
    print(ch_event)
    log_tree_structure(ch)

In [ ]:
node.rebuild_children()
node.build_tree()
node.convert_to_full_up_to_depth(np.inf)

In [ ]:
print(node.children_events)
print([f"{ch.get_value():.2f}" for ch in node.children])

for ch, ch_event in zip(node.children, node.children_events):
    print(ch_event)
    log_tree_structure(ch)

In [ ]:
root_node.convert_to_full_up_to_depth(depth=np.inf)

In [ ]:
values = []

values.append(root_node.get_value())

for i in tqdm.tqdm(range(1000)):
    root_node.resample_player_cards()
    values.append(root_node.get_value())
mean_value = np.mean(values)


In [ ]:
f, ax = plt.subplots()
ax.scatter(range(len(values)), values)
ax.hlines(mean_value, xmin=0, xmax=len(values), color="red", label=f"mean={mean_value:.3f}")
ax.legend()

In [ ]:
np.min(values), np.max(values)

In [ ]:
for lvl in range(25):
    level_nodes = get_nodes_on_the_level(root_node, lvl)
    print(f"Level {lvl} has {len(level_nodes)} nodes")
    finished = False
    for n in level_nodes:
        if n.bj_round.get_stage() == BJStage.ROUND_OVER:
            continue
        elif isinstance(n, MonteCarloNode):
            print(f"MonteCarloNode found on level {lvl}")
            finished = True
            break
        else:
            break
    
    if finished:
        break

In [ ]:
from collections import Counter
count = Counter()

player_card_nodes = []
player_action_nodes = []
for node in level_nodes:
    stage = node.bj_round.get_stage()
    count[stage] += 1
    if stage == BJStage.PLAYER_CARD:
        player_card_nodes.append(node)
    if stage == BJStage.PLAYER_ACTION:
        player_action_nodes.append(node)



In [ ]:
count

In [ ]:
for i, n in enumerate(player_action_nodes):
    print(f"idx = {i}")
    print(f"Value = {n.get_value()}")
    print(str(n.bj_round))
    print()

In [ ]:
this_node = player_card_nodes[-1].parent
this_node.build_tree()

print(f"this_node Value = {this_node.get_value()}")
print(str(this_node.bj_round))
first_value = this_node.get_value()

In [ ]:
for i in range(100):
    changed = this_node.resample_player_cards()
    if this_node.get_value() == first_value:
        continue
    print(changed, this_node.get_value())
    print("-" * 32)

    # for i in range(len(this_node.children)):
    #     p = this_node.children_prob[i]
    #     ch = this_node.children[i]
    #     print(f"Value = {ch.get_value()}")
    #     print(f"Probability = {p}")
    #     print(str(ch.bj_round))
    #     print()
    # print("-" * 32)


In [ ]:
# this_node.has_completed_tree = False
# this_node.has_built_children = False
# this_node.children_prob = []
# this_node.children = []

In [ ]:
print(this_node.children[1].children[-1].bj_round)

In [ ]:
for i in range(100):
    this_node.resample_player_cards()
    children_of_interest = this_node.children[1].children[-1].children

    # if this_node.children_prob[1] == 0:
    #    continue
    
    # print(this_node.get_value())
    # print([f"{ch.get_value():.2f}" for ch in this_node.children])
    print(this_node.children_prob)

In [ ]:
this_node.resample_player_cards()

for ch in this_node.children:
    print(ch.get_value())
    print(str(ch.bj_round))
    print()

In [ ]:

print(this_node.children[0].get_value())

In [ ]:
print(this_node.children)